<div align="right"><a href="L34-Burg-Algorithmus.ipynb">Switch to German language 🇩🇪</a></div>

Die folgenden Übungen sind Teil des Buches
* K.-D. Kammeyer, K. Kroschel: "Digitale Signalverarbeitung - Filterung und Spektralanalyse", Springer Vieweg, 2022.  [[KK22]](https://doi.org/10.1007/978-3-663-09805-8)

---

This Jupyter Notebook can be opened in [Google Colab](https://colab.research.google.com/notebooks/intro.ipynb) by clicking on the icon on the right, but this notebook is displayed best when opened locally using [Jupyter Notebook](https://jupyter.org/) (e.g. as part of the [Anaconda](https://www.anaconda.com/) distribution). <a href="https://colab.research.google.com/github/Stefan-Goe/dsp-book/blob/main/python/S34-Burg-Algorithm.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open Notebook in Google Colab"></a>

---

<!--NAVIGATION-->
<div align="center">

Previous: [S33 Yule-Walker-/Autokorrelationsmethode](./S33-Yule-Walker-und-Autokorrelationsmethode.ipynb) | [Exercise 34](./E34-Burg-Algorithm.ipynb) | [Contents / Overview](./00-Python-Notebooks-Overview-EN.ipynb) | Next: [S35 Comparison of Spectral Estimators](./S35-SpectralEstimators.ipynb)

</div>

---


## Exercise: Burg

The basic idea of the Burg algorithm is to use the balance of the power  
of the forward and backward prediction error  
and to minimize both powers jointly.  
This approach yields an unbiased estimation of the coefficients of the AR($n$)-model  
(in contrast to the autocorrelation method); additionally, the analysis filter  
is always a minimum-phase filter (in contrast to the autocovariance method).

We start with an (all-pole) system  

$H(z) = \frac{1}{A(z)}$  

with  

$$
A(z) = 1 + 0.3611z^{-1} - 0.1373z^{-2} + 0.2685z^{-3}
- 0.0533z^{-4} - 0.3353z^{-5} - 0.307z^{-6}
$$

of order $m = 6$, which is excited by a stationary, zero-mean, white noise process  
$N(k)$ with power $\sigma_N^2 = 1$  
(using, e.g., `scipy.signal.lfilter`).

<img src="./pngs/l15sys.eps" style="width:70%;">

---

### a)

Plot the impulse response $h(k)$ of the system in the interval  

$$
0 \le k \le 80
$$

using an impulse response computation  
(e.g. `scipy.signal.dimpulse`)  
and plot the pole-zero diagram using the provided `zplane` function  
(from `helper_functions.py`).

---

### b)

Estimate the coefficients of an AR-system AR($n$)

$$
\frac{1}{A_n(z)} = \frac{1}{a_0 + a_1 z^{-1} + \cdots + a_n z^{-n}}
$$

based on $N = 2^{13}$ samples of the process $X(k)$  
for different model orders $n = 2,\; 6,\; 32$  
(e.g. using a Burg algorithm implementation available in Python).

Compare the different estimated power spectral densities with each other  
and with the exact power spectral density  
$S_{XX}(e^{j\Omega})$.

Why is the estimation particularly good for $n = 6$?

---

### c)

Repeat exercises a) and b) for a FIR system

$$
H(z) = 1 - 4z^{-1} + 6z^{-2} - 4z^{-3} + z^{-4}
$$

of order $m = 4$.

Observe the arrangement of the poles of the AR($n$)-model  
used to compensate the fourfold zero of $H(z)$ at $z = 1$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from helper_functions import zplane, burg_algo
# BEGIN SOLUTION
# ##############################################################################
# ##  Subtask a: AR-System, Transient Response                                ##
# ##############################################################################

MA = np.array([1.0])
AR = np.array([1.0000, 0.3611, -0.1373, 0.2685, -0.0533, -0.3353, -0.307])
NFFT = 2**10

# Impulse response (impz)
t = np.arange(80)
impulse = np.zeros(80)
impulse[0] = 1.0
h_arma = signal.lfilter(MA, AR, impulse)

# Graphical output
plt.figure(figsize=(10, 4))
plt.stem(t, h_arma)
plt.grid(True)
plt.xlabel('k')
plt.ylabel('h(k)')
plt.title(f'Impulse Response of the AR({len(AR)-1}) Model')

# Pole-Zero plot
zplane(MA, AR, title=f'Poles of the AR({len(AR)-1}) Model')
plt.show()

# ##############################################################################
# ##  Subtask b: AR-System, Power Spectral Density (PSD)                      ##
# ##############################################################################

Omega_norm = np.linspace(0, 1, NFFT, endpoint=False)
# Exact PSD calculation
w, h = signal.freqz(MA, AR, worN=NFFT, whole=True)
Sxx_arma = np.abs(h)**2

n_orders = [2, len(AR)-1, 32]
N = 2^13
N_pre = 80 # Transient samples to discard
noise = np.random.randn(N + N_pre)
x_all = signal.lfilter(MA, AR, noise)
x = x_all[N_pre:]

for n in n_orders:
    ar_est = burg_algo(x, n)
    w_est, h_est = signal.freqz([1], ar_est, worN=NFFT, whole=True)
    Sxx_ar = np.abs(h_est)**2
    
    # Normalization as in the MATLAB code
    Sxx_ar = Sxx_ar / np.sum(Sxx_ar) * np.sum(Sxx_arma)
    
    # Plot PSD
    plt.figure()
    plt.plot(Omega_norm, 10 * np.log10(Sxx_arma / np.max(Sxx_arma)), 'r--', label='Exact PSD')
    plt.plot(Omega_norm, 10 * np.log10(Sxx_ar / np.max(Sxx_ar)), 'b-', label='Estimated PSD')
    plt.ylabel('$S_{xx}(e^{j\Omega})$ in dB')
    plt.xlabel('$\Omega/2\pi$')
    plt.title(f'PSD of Burg-AR({n}), N={N}')
    plt.grid(True)
    plt.legend()
    plt.show()
    
    # Plot Poles for the estimation
    zplane([1], ar_est, title=f'Poles of Burg-AR({n})')
    plt.show()

# ##############################################################################
# ##  Subtask c: MA-System                                                    ##
# ##############################################################################

MA_c = np.array([1, -4, 6, -4, 1])
AR_c = np.array([1.0])

# Impulse response
h_arma_c = signal.lfilter(MA_c, AR_c, np.concatenate(([1], np.zeros(len(MA_c)-1))))
t_c = np.arange(len(h_arma_c))

plt.figure()
plt.stem(t_c, h_arma_c)
plt.grid(True)
plt.xlabel('k')
plt.ylabel('h(k)')
plt.title(f'Impulse Response of the MA({len(MA_c)-1}) Model')

# Pole-Zero plot for MA model
zplane(MA_c, AR_c, title=f'Poles/Zeros of the MA({len(MA_c)-1}) Model')
plt.show()

# Analyze filter order AR(n) for the MA system (Spectral Estimation)
w_c, h_c = signal.freqz(MA_c, AR_c, worN=NFFT, whole=True)
Sxx_arma_c = np.abs(h_c)**2

N_pre_c = 5
noise_c = np.random.randn(N + N_pre_c)
x_all_c = signal.lfilter(MA_c, AR_c, noise_c)
x_c = x_all_c[N_pre_c:]

for n in [2, 6, 32]:
    ar_est_c = burg_algo(x_c, n)
    w_est_c, h_est_c = signal.freqz([1], ar_est_c, worN=NFFT, whole=True)
    Sxx_ar_c = np.abs(h_est_c)**2
    
    Sxx_ar_c = Sxx_ar_c / np.sum(Sxx_ar_c) * np.sum(Sxx_arma_c)
    
    # Avoid log(0) for plotting
    Sxx_ar_c[Sxx_ar_c == 0] = 1e-10
    Sxx_arma_c[Sxx_arma_c == 0] = 1e-10
    
    # Plot PSD comparison
    plt.figure()
    plt.plot(Omega_norm, 10 * np.log10(Sxx_arma_c / np.max(Sxx_arma_c)), 'r--', label='Exact PSD')
    plt.plot(Omega_norm, 10 * np.log10(Sxx_ar_c / np.max(Sxx_ar_c)), 'b-', label='Estimated PSD')
    plt.ylim([-70, 0])
    plt.grid(True)
    plt.ylabel('$S_{xx}(e^{j\Omega})$ in dB')
    plt.xlabel('$\Omega/2\pi$')
    plt.title(f'PSD of Burg-AR({n}) (MA Source), N={N}')
    plt.legend()
    plt.show()
    
    # Plot Poles of the AR approximation
    zplane([1], ar_est_c, title=f'Poles of Burg-AR({n})')
    plt.show()
# END SOLUTION

## Solution: Burg

### a)

The following figure shows the impulse response $h(k)$ of the AR(6) system  
for the interval $0 \le k \le 80$  
as well as the corresponding pole–zero plot,  
computed using standard Python signal-processing routines  
(for the pole–zero plot, the provided `zplane` function is used),  
with poles at  

- $z_{\infty,1} = 0.3444 + i\cdot 0.8315$  
- $z_{\infty,2} = z_{\infty,1}^{*}$  
- $z_{\infty,3} = 0.85$  
- $z_{\infty,4} = -0.91$  
- $z_{\infty,5} = -0.49 + i\cdot 0.4950$  
- $z_{\infty,6} = z_{\infty,5}^{*}$  

<div style="display:flex; justify-content:center; gap:20px;">
  <img src="./pngs/ldsv9a03_a1.png" style="width:45%;">
  <img src="./pngs/ldsv9a03_a2.png" style="width:45%;">
</div>

---

### b)

Procedures for spectrum estimation (using the Yule–Walker or Burg method)  
aim to model an AR-filtered white noise process  
(i.e. to estimate the AR parameters  
$\hat{a}_{0,n}, \ldots, \hat{a}_{n,n}$),  
such that its statistical properties (in particular the autocorrelation function)  
closely match those of the given stationary process $X(k)$ with arbitrary spectrum.

This approach works best if the underlying process is indeed generated  
by AR filtering and if the order of the AR($n$) model matches the order  
of the true filter.  
The case of filtering with an AR system of order $m = 6$ is shown below.

<div style="display:flex; justify-content:center; gap:20px;">
  <img src="./pngs/ldsv9a03_b11.png" style="width:45%;">
  <img src="./pngs/ldsv9a03_b12.png" style="width:45%;">
</div>

The two upper figures show the exact power spectral density compared with  
power spectral densities obtained from AR coefficients estimated using  
the Burg algorithm with model orders $n = 2, 6, 32$.

The lower figures depict the corresponding pole locations;  
the poles of the original system $H(z)$ are explicitly marked as *circles*.  
All computations are based on a single realization of length  
$N = 2^{13} = 8192$, which provides a statistically sufficient estimation.

If the AR model order is underestimated (e.g. $n = 2$),  
the Burg algorithm attempts to find the best possible compromise  
in the pole locations.  
Nevertheless, a significant deviation between the estimated  
and the exact power spectral density remains.

<div style="display:flex; justify-content:center; gap:20px;">
  <img src="./pngs/ldsv9a03_b21.png" style="width:45%;">
  <img src="./pngs/ldsv9a03_b22.png" style="width:45%;">
</div>

If the order of the AR($n$) model matches the order of the true filter $H(z)$,  
the approximation quality is—as expected—the best.

<div style="display:flex; justify-content:center; gap:20px;">
  <img src="./pngs/ldsv9a03_b31.png" style="width:45%;">
  <img src="./pngs/ldsv9a03_b32.png" style="width:45%;">
</div>

In the case of overestimation with $n = 32$, all additional coefficients  
are estimated to be nearly zero:

$$
\hat{a}_{n,i} \approx 0 \quad \text{for} \quad i > m
$$

These coefficients correspond to poles placed on a circle close to  
the unit circle. As a result, the estimated power spectral density  
exhibits a more “wavy” behavior.  
However, the degradation in approximation quality remains small.

---

### c)

The following figure shows the impulse response $h(k)$ of the MA(4) system  
for the interval $0 \le k \le 4$.

Since $H(z)$ is an MA model of order $m = 4$,  
the impulse response has finite length $m + 1 = 5$:

$$
h(k) = \{1, -4, 6, -4, 1\}
$$

Consequently, the energy correlation function  
$r_{HH}^{(E)}(\kappa)$ and thus the autocorrelation function  
$r_{XX}(\kappa)$  
are limited to the range $|\kappa| \le 4$.

This yields the values:

- $r_{XX}(0) = 70$  
- $r_{XX}(\pm 1) = -56$  
- $r_{XX}(\pm 2) = 28$  
- $r_{XX}(\pm 3) = -8$  
- $r_{XX}(\pm 4) = 1$  
- $r_{XX}(\kappa) = 0$ for $|\kappa| \ge 5$

Note that $H(z)$ has a fourfold zero at $z = 1$,  
corresponding to high-pass behavior.

<div style="display:flex; justify-content:center; gap:20px;">
  <img src="./pngs/ldsv9a03_c1.png" style="width:45%;">
  <img src="./pngs/ldsv9a03_c2.png" style="width:45%;">
</div>

---

### d)

Using the Burg algorithm implemented in Python,  
the MA model $H(z)$ is approximated by AR models  
$1/A_n(z)$ of different orders $n = 2, 6, 32$.

<div style="display:flex; justify-content:center; gap:20px;">
  <img src="./pngs/ldsv9a03_d11.png" style="width:45%;">
  <img src="./pngs/ldsv9a03_d12.png" style="width:45%;">
</div>

<div style="display:flex; justify-content:center; gap:20px;">
  <img src="./pngs/ldsv9a03_d21.png" style="width:45%;">
  <img src="./pngs/ldsv9a03_d22.png" style="width:45%;">
</div>

<div style="display:flex; justify-content:center; gap:20px;">
  <img src="./pngs/ldsv9a03_d31.png" style="width:45%;">
  <img src="./pngs/ldsv9a03_d32.png" style="width:45%;">
</div>

The results show that the Burg algorithm approximates zeros  
by placing a constellation of poles close to the unit circle.

For low model order ($n = 2$), the poles are placed as far away  
from the zero as possible and arranged on a circle directed toward it,  
while the immediate vicinity of the zero remains free.

From the estimated power spectral densities it can be observed that  
a satisfactory approximation is achieved when higher-order AR models  
are used.